In [ ]:
# import libraries
import urllib.request
import PIL.Image
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
# reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# extract data
imgURL = "https://raw.githubusercontent.com/larvalabs/cryptopunks/master/punks.png"
urllib.request.urlretrieve(imgURL, "cryptoPunksAll.png")
img = PIL.Image.open("cryptoPunksAll.png").convert("RGB")
imgArray = np.asarray(img)

In [5]:
# create dataset
n = 10000
finalArray = np.empty((n, 24, 24, 3), dtype=np.float32)

for i in range(100):
    for j in range(100):
        a, b = 24 * i, 24 * (i + 1)
        c, d = 24 * j, 24 * (j + 1)
        idx = j + i * 100
        finalArray[idx, :, :, :] = imgArray[a:b, c:d, :]
        
d1min, d1max = 11, 17
d2min, d2max = 9, 14

cancerpunks = finalArray.copy()
label = np.zeros(n, dtype=np.float32)

for i in range(n):
    if np.random.randint(0, 2) == 1:
        label[i] = 1.0
        d1loc = np.random.randint(d1min, d1max + 1)
        d2loc = np.random.randint(d2min, d2max + 1)
        cancerpunks[i, d1loc, d2loc, :] = 255.0

In [9]:
# pytorch expectaions / normalized to [0,1]
X = np.transpose(cancerpunks / 255.0, (0, 3, 1, 2))
y = label

In [8]:
# create training and validation sets
indices = np.random.permutation(n)
split = int(0.8 * n)

X_train = torch.tensor(X[indices[:split]])
y_train = torch.tensor(y[indices[:split]])
X_val = torch.tensor(X[indices[split:]])
y_val = torch.tensor(y[indices[split:]])

print(f"Training set: {len(X_train)} images ({y_train.mean().item():.1%} positive)")
print(f"Validation set: {len(X_val)} images ({y_val.mean().item():.1%} positive)\n")

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

Training set: 8000 images (50.6% positive)
Validation set: 2000 images (50.1% positive)



In [ ]:
# build the CNN
model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(64, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(64 * 6 * 6, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

In [12]:
# train the CNN
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

epochs = 15
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb).squeeze(1), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(xb)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} - Loss: {running_loss / len(X_train):.4f}")

Epoch 01/15 - Loss: 0.6934
Epoch 05/15 - Loss: 0.0944
Epoch 10/15 - Loss: 0.0417
Epoch 15/15 - Loss: 0.0190
